In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).
✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- Cell 22: V42 (Per-Lead Lengths + Per-Lead FS + Safe Gating) ---
import os, csv, glob, gc, math
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# ---------------------------
# 0) Constants
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"
TEST_CSV_PATH = f"{IMAGE_DIR}/test.csv"
SAMPLE_PARQUET_PATH = f"{IMAGE_DIR}/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

YOLO_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_TO_IDX = {name: i for i, name in enumerate(LEAD_NAMES)}

TARGET_H = 256
YOLO_INF_MAX = 1280
YOLO_CONF = 0.10
MAX_W = 2048
MAX_CACHE = 16

# Viterbi
DP_MAX_W = 1400
DP_SMOOTH = 0.45

# Quality gate (حتى ما ترجع أسوأ من zeros)
MIN_MEAN_PROB = 0.03          # إذا متوسط احتمالات الماسك منخفض جدًا => lead فاشل
MIN_P2P_MV = 0.05             # أقل سعة مقبولة بعد التحويل
MAX_P2P_MV = 12.0             # أعلى سعة مقبولة

print(f"⚡ Device: {DEVICE}")

# ---------------------------
# 1) Read template ids
# ---------------------------
print("📦 Reading Parquet template ids...")
template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = template["id"].astype(str).to_numpy()
del template
gc.collect()
print(f"✅ Template rows: {len(template_ids):,}")

# ---------------------------
# 2) Read test.csv => per-lead lengths + per-lead fs
# ---------------------------
print("🧠 Reading test.csv (per-lead lengths + fs)...")
tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)

# normalize columns safely
cols = {c.lower(): c for c in tdf.columns}
id_c = cols.get("id", None)
lead_c = cols.get("lead", None)
fs_c = cols.get("fs", None)
nrows_c = cols.get("number_of_rows", None)

assert id_c and lead_c and fs_c and nrows_c, "❌ test.csv columns not as expected"

tdf[id_c] = tdf[id_c].astype(str).str.replace(r"\.0$", "", regex=True)
tdf[lead_c] = tdf[lead_c].astype(str).str.strip()
tdf[fs_c] = tdf[fs_c].astype(float)
tdf[nrows_c] = tdf[nrows_c].astype(int)

len_map = {}   # (pid, lead) -> nrows
fs_map  = {}   # (pid, lead) -> fs
for pid, lead, fs, nrows in zip(tdf[id_c], tdf[lead_c], tdf[fs_c], tdf[nrows_c]):
    key = (str(pid), str(lead))
    len_map[key] = int(nrows)
    fs_map[key] = float(fs)

print(f"✅ len_map entries: {len(len_map)}")
print(f"✅ unique patients in test.csv: {len(set([k[0] for k in len_map.keys()]))}")
print("🔎 Example:", list(len_map.items())[:4])

# ---------------------------
# 3) Index images
# ---------------------------
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path(pid):
    pid_s = clean_pid(pid)
    if pid_s in path_map:
        return path_map[pid_s]
    try:
        return path_map.get(str(int(float(pid_s))))
    except:
        return None

def safe_read_rgb(path):
    try:
        img = cv2.imread(path)
        if img is None:
            return None
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except:
        return None

# ---------------------------
# 4) Load models + YOLO mapping
# ---------------------------
print("⚙️ Loading models (offline)...")

yolo_model = None
CLASS_TO_LEAD_IDX = {}

def _normalize_lead_name(x: str) -> str:
    s = str(x).strip()
    s = s.replace("Lead", "").replace("lead", "").replace("_", "").replace("-", "").replace(" ", "")
    s = s.replace("AVR", "aVR").replace("AVL", "aVL").replace("AVF", "aVF")
    s = s.replace("AVr","aVR").replace("AVl","aVL").replace("AVf","aVF")
    return s

if os.path.exists(YOLO_PATH):
    yolo_model = YOLO(YOLO_PATH)
    print(f"✅ YOLO loaded: {YOLO_PATH}")
    names = getattr(yolo_model, "names", None)
    items = list(names.items()) if isinstance(names, dict) else list(enumerate(names)) if isinstance(names, list) else []
    for cid, cname in items:
        n = _normalize_lead_name(cname)
        if n in LEAD_TO_IDX:
            CLASS_TO_LEAD_IDX[int(cid)] = LEAD_TO_IDX[n]

    # لو الماب ناقص: لا تعمل identity بشكل أعمى — لكن نخلي fallback محدود
    for i in range(12):
        CLASS_TO_LEAD_IDX.setdefault(i, i)

    # اطبع أسماء YOLO للتأكد (مهم جدًا)
    print("🧾 YOLO names:", names)
    print(f"✅ CLASS_TO_LEAD_IDX: {CLASS_TO_LEAD_IDX}")
else:
    print("⚠️ YOLO_PATH not found!")

unet_model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    decoder_attention_type="scse",
)
if os.path.exists(UNET_PATH):
    unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
    print(f"✅ UNet loaded: {UNET_PATH}")
else:
    print("⚠️ UNET_PATH not found!")

unet_model.to(DEVICE).eval()

# ---------------------------
# 5) Helpers
# ---------------------------
def preprocess_remove_grid_rgb(img_rgb):
    try:
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        out = img_rgb.copy()
        out[mask > 0] = (255, 255, 255)
        return out
    except:
        return img_rgb

def apply_high_pass(data, cutoff=0.5, fs=500.0, order=3):
    try:
        if len(data) < order * 3:
            return data
        nyq = 0.5 * fs
        if nyq <= 0:
            return data
        wn = cutoff / nyq
        if not (0 < wn < 1):
            return data
        b, a = butter(order, wn, btype="high")
        return filtfilt(b, a, data)
    except:
        return data

# ---- grid estimate + ppmv chooser ----
def estimate_grid_spacing_px(img_rgb):
    try:
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        if gray.size == 0 or np.std(gray) < 3:
            return None
        ex = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        ey = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sx = np.sum(np.abs(ex), axis=0)
        sy = np.sum(np.abs(ey), axis=1)
        px, _ = find_peaks(sx, distance=10, prominence=np.percentile(sx, 75))
        py, _ = find_peaks(sy, distance=10, prominence=np.percentile(sy, 75))
        diffs = []
        if len(px) > 5:
            dx = np.diff(px); diffs.extend(dx[(dx > 8) & (dx < 120)])
        if len(py) > 5:
            dy = np.diff(py); diffs.extend(dy[(dy > 8) & (dy < 120)])
        if len(diffs) < 6:
            return None
        return float(np.median(np.array(diffs)))
    except:
        return None

def choose_ppmv(local_grid_px, resize_scale, raw_trace_px):
    if local_grid_px is None or not np.isfinite(local_grid_px) or local_grid_px < 5:
        return 250.0

    # A: measured = 1mm => ppmv = grid*10
    # B: measured = 5mm => ppmv = grid*2
    ppmv_A = (local_grid_px * 10.0) * resize_scale
    ppmv_B = (local_grid_px * 2.0)  * resize_scale

    def score(ppmv):
        if ppmv <= 0: return 1e9
        sig = (raw_trace_px - np.median(raw_trace_px)) / ppmv
        sig = np.nan_to_num(sig)
        p2p = float(np.percentile(sig, 99) - np.percentile(sig, 1))
        if p2p <= 0: return 1e9
        if p2p < 0.2: return 1000.0 + (0.2 - p2p) * 1500.0
        if p2p > 8.0: return 1000.0 + (p2p - 8.0) * 400.0
        return abs(p2p - 2.5)

    sA, sB = score(ppmv_A), score(ppmv_B)
    ppmv = ppmv_A if sA <= sB else ppmv_B
    if not np.isfinite(ppmv) or ppmv < 10:
        return 250.0
    return float(ppmv)

# ---------------------------
# 6) Viterbi (Adaptive)
# ---------------------------
def viterbi_dp(prob_map, smooth=0.45):
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6).astype(np.float32)
    dp = np.zeros_like(cost, dtype=np.float32)
    parent = np.zeros((H, W), dtype=np.int16)

    dp[:, 0] = cost[:, 0]
    parent[:, 0] = np.arange(H, dtype=np.int16)

    for t in range(1, W):
        prev = dp[:, t-1]
        c_m1 = np.roll(prev, 1);  c_m1[0] = 1e9
        c_0  = prev
        c_p1 = np.roll(prev, -1); c_p1[-1] = 1e9

        stacked = np.stack([c_m1 + smooth, c_0, c_p1 + smooth], axis=0)
        which = np.argmin(stacked, axis=0).astype(np.int16)
        best_prev = stacked[which, np.arange(H)]

        dp[:, t] = cost[:, t] + best_prev
        parent[:, t] = np.clip(np.arange(H, dtype=np.int16) + (which - 1), 0, H-1)

    path = np.zeros(W, dtype=np.int32)
    path[-1] = int(np.argmin(dp[:, -1]))
    for t in range(W-2, -1, -1):
        path[t] = int(parent[path[t+1], t+1])

    return (H - path).astype(np.float32)

def viterbi_adaptive(prob_map):
    H, W = prob_map.shape
    if W <= 20 or H <= 5:
        path = np.argmax(prob_map, axis=0)
        return (H - path).astype(np.float32)

    if W <= DP_MAX_W:
        return viterbi_dp(prob_map, smooth=DP_SMOOTH)

    path = np.argmax(prob_map, axis=0).astype(np.float32)
    win = 21 if W >= 21 else (W if W % 2 == 1 else W-1)
    if win >= 5:
        path = savgol_filter(path, win, 2)
    return (H - path).astype(np.float32)

# ---------------------------
# 7) YOLO crops (return crop + conf per lead)
# ---------------------------
def get_crops_yolo_mapped(img_rgb):
    crops = [None] * 12
    confs = [0.0] * 12
    if yolo_model is None:
        return crops, confs

    h0, w0 = img_rgb.shape[:2]
    if h0 < 20 or w0 < 20:
        return crops, confs

    scale = YOLO_INF_MAX / float(max(h0, w0))
    if scale < 1.0:
        w1, h1 = max(32, int(w0 * scale)), max(32, int(h0 * scale))
        img_inf = cv2.resize(img_rgb, (w1, h1))
    else:
        img_inf = img_rgb
        scale = 1.0

    best = {}
    try:
        res = yolo_model.predict(img_inf, conf=YOLO_CONF, verbose=False, device=DEVICE)
        if res and res[0].boxes is not None and len(res[0].boxes) > 0:
            boxes = res[0].boxes.data.detach().cpu().numpy()
            for b in boxes:
                x1, y1, x2, y2, conf, cls_id = b[:6]
                cls_id = int(cls_id)
                conf = float(conf)
                lead_idx = CLASS_TO_LEAD_IDX.get(cls_id, cls_id)
                if not (0 <= lead_idx < 12):
                    continue

                x1o, y1o, x2o, y2o = x1 / scale, y1 / scale, x2 / scale, y2 / scale
                x1o, y1o = max(0, int(x1o)), max(0, int(y1o))
                x2o, y2o = min(w0, int(x2o)), min(h0, int(y2o))
                if x2o <= x1o + 10 or y2o <= y1o + 10:
                    continue

                prev = best.get(lead_idx)
                if (prev is None) or (conf > prev[0]):
                    best[lead_idx] = (conf, (x1o, y1o, x2o, y2o))

        for li, (cf, (x1o, y1o, x2o, y2o)) in best.items():
            crops[li] = img_rgb[y1o:y2o, x1o:x2o]
            confs[li] = float(cf)

        return crops, confs
    except:
        return crops, confs

# ---------------------------
# 8) UNet batch predict
# ---------------------------
def prepare_unet_batch(crops_rgb):
    tens_list, scales, widths = [], [], []
    for c in crops_rgb:
        h, w = c.shape[:2]
        if h < 5 or w < 5:
            continue
        sc = TARGET_H / float(h)
        nw = int(max(1, w * sc))
        if nw > MAX_W:
            nw = MAX_W
            sc = nw / float(w)

        rz = cv2.resize(c, (nw, TARGET_H))
        t = torch.from_numpy(rz).permute(2, 0, 1).float() / 255.0
        tens_list.append(t)
        scales.append(sc)
        widths.append(nw)

    if not tens_list:
        return None, None, None

    max_w = int(np.ceil(max(widths) / 32.0) * 32)
    max_w = min(max_w, MAX_W)

    batch = torch.zeros((len(tens_list), 3, TARGET_H, max_w), dtype=torch.float32, device=DEVICE)
    for i, t in enumerate(tens_list):
        w_i = min(t.shape[2], max_w)
        batch[i, :, :, :w_i] = t[:, :, :w_i].to(DEVICE)

    return batch, scales, widths

def predict_unet_probmaps(crops_rgb):
    batch, scales, widths = prepare_unet_batch(crops_rgb)
    if batch is None:
        return [], []

    try:
        with torch.no_grad():
            if DEVICE == "cuda":
                with torch.amp.autocast("cuda", enabled=True):
                    p1 = torch.sigmoid(unet_model(batch))
                    p2 = torch.sigmoid(unet_model(torch.flip(batch, [3])))
                    preds = (p1 + torch.flip(p2, [3])) / 2.0
            else:
                p1 = torch.sigmoid(unet_model(batch))
                p2 = torch.sigmoid(unet_model(torch.flip(batch, [3])))
                preds = (p1 + torch.flip(p2, [3])) / 2.0

        preds = preds.detach().float().cpu().numpy()  # [N,1,H,W]
        prob_maps = []
        for i in range(preds.shape[0]):
            w_i = widths[i]
            prob_maps.append(preds[i, 0, :, :w_i].astype(np.float32))

        return prob_maps, scales

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return [], []
        raise

# ---------------------------
# 9) Compute patient leads (IMPORTANT: per-lead length!)
# ---------------------------
patient_cache = OrderedDict()

def compute_patient_leads(pid):
    """
    returns list of 12 arrays, each array length = number_of_rows for that (pid, lead)
    missing/failed leads => zeros (same length)
    """
    pid = clean_pid(pid)
    # prepare expected meta per lead
    exp_len = []
    exp_fs  = []
    for ln in LEAD_NAMES:
        L = int(len_map.get((pid, ln), 5000))
        F = float(fs_map.get((pid, ln), 500.0))
        exp_len.append(L)
        exp_fs.append(F)

    # default outputs = zeros with correct per-lead lengths
    out = [np.zeros(L, dtype=np.float32) for L in exp_len]

    path = get_image_path(pid)
    if not path:
        return out

    img = safe_read_rgb(path)
    if img is None:
        return out

    crops, yconfs = get_crops_yolo_mapped(img)
    detected = [(i, crops[i], yconfs[i]) for i in range(12) if crops[i] is not None and crops[i].size > 200]

    if not detected:
        return out  # better than wrong fallback

    crops_rgb = [preprocess_remove_grid_rgb(c) for _, c, _ in detected]
    prob_maps, scales = predict_unet_probmaps(crops_rgb)
    if not prob_maps:
        return out

    global_grid = estimate_grid_spacing_px(img)

    for j in range(len(prob_maps)):
        lead_idx, crop_orig, yconf = detected[j]
        prob = prob_maps[j]
        sc = scales[j]
        fs = exp_fs[lead_idx]
        target_len = exp_len[lead_idx]

        # quality gate 1: prob-map too weak
        if float(np.mean(prob)) < MIN_MEAN_PROB:
            continue

        raw_px = viterbi_adaptive(prob)

        local_grid = estimate_grid_spacing_px(crop_orig)
        if local_grid is None:
            local_grid = global_grid

        ppmv = choose_ppmv(local_grid, sc, raw_px)
        sig_mv = (raw_px - np.median(raw_px)) / ppmv
        sig_mv = np.nan_to_num(sig_mv, nan=0.0, posinf=0.0, neginf=0.0)

        # mild smoothing + (اختياري) high-pass
        if len(sig_mv) >= 11:
            sig_mv = savgol_filter(sig_mv, 11, 3)
        if len(sig_mv) >= 30:
            sig_mv = apply_high_pass(sig_mv, cutoff=0.5, fs=fs, order=3)

        # quality gate 2: amplitude plausibility
        p2p = float(np.percentile(sig_mv, 99) - np.percentile(sig_mv, 1))
        if not (MIN_P2P_MV <= p2p <= MAX_P2P_MV):
            continue

        sig_mv = np.clip(sig_mv, -7.0, 7.0)

        # ✅ RESAMPLE TO THIS LEAD'S OWN EXPECTED LENGTH
        out[lead_idx] = resample(sig_mv, target_len).astype(np.float32)

    return out

# ---------------------------
# 10) Streaming write (per-lead variable lengths)
# ---------------------------
print("🚀 Writing submission.csv (V42 Per-Lead Lengths)...")
with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing Rows"):
        try:
            pid_raw, idx_raw, lead_name = rid.rsplit("_", 2)
            pid = clean_pid(pid_raw)
            idx = int(idx_raw)
            lead_idx = LEAD_TO_IDX.get(lead_name, 0)

            # cache per patient
            if pid in patient_cache:
                sigs = patient_cache[pid]
                patient_cache.move_to_end(pid)
            else:
                sigs = compute_patient_leads(pid)
                patient_cache[pid] = sigs
                if len(patient_cache) > MAX_CACHE:
                    patient_cache.popitem(last=False)

            sig = sigs[lead_idx]
            val = 0.0
            if 0 <= idx < len(sig):
                v = float(sig[idx])
                if np.isfinite(v):
                    val = v

            writer.writerow([rid, f"{val:.4f}"])

        except:
            writer.writerow([rid, "0.0000"])

del patient_cache
gc.collect()
torch.cuda.empty_cache()
print("✅ Done. submission.csv ready.")


⚡ Device: cuda
📦 Reading Parquet template ids...
✅ Template rows: 75,000
🧠 Reading test.csv (per-lead lengths + fs)...
✅ len_map entries: 24
✅ unique patients in test.csv: 2
🔎 Example: [(('1053922973', 'I'), 2500), (('1053922973', 'II'), 10000), (('1053922973', 'III'), 2500), (('1053922973', 'aVR'), 2500)]
🗂️ Indexing images...
✅ Indexed images: 8,795
⚙️ Loading models (offline)...
✅ YOLO loaded: /kaggle/input/ecg-final-models/best.pt
🧾 YOLO names: {0: 'I', 1: 'II', 2: 'III', 3: 'aVR', 4: 'aVL', 5: 'aVF', 6: 'V1', 7: 'V2', 8: 'V3', 9: 'V4', 10: 'V5', 11: 'V6'}
✅ CLASS_TO_LEAD_IDX: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}
✅ UNet loaded: /kaggle/input/ecg-final-models/best_model_phase10.pth
🚀 Writing submission.csv (V42 Per-Lead Lengths)...


Processing Rows: 100%|██████████| 75000/75000 [00:03<00:00, 21399.45it/s]


✅ Done. submission.csv ready.


In [5]:
# =========================
# --- Cell 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

sub = pd.read_csv(SUBMISSION_FILE)

assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.95 MB
🎉 Ready to Submit.


In [6]:
print(len(tdf), tdf['id'].nunique())

24 2
